In [8]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

In [ ]:


class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
graph.invoke({"foo": "", "bar":[]}, config)

{'foo': 'b', 'bar': 'b'}

After we run the graph, we expect to see exactly 4 checkpoints:
    Empty checkpoint with START as the next node to be executed
    Checkpoint with the user input {'foo': '', 'bar': []} and node_a as the next node to be executed
    Checkpoint with the outputs of node_a {'foo': 'a', 'bar': ['a']} and node_b as the next node to be executed
    Checkpoint with the outputs of node_b {'foo': 'b', 'bar': ['a', 'b']} and no next nodes to be executed

In [12]:
# # get the latest state snapshot
# config = {"configurable": {"thread_id": "1"}}
# graph.get_state(config)

# get a state snapshot for a specific checkpoint_id
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1f0dc245-364d-6625-8002-5acb00f7c405"}}
graph.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': 'b'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0dc245-364d-6625-8002-5acb00f7c405'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-12-18T15:15:02.453610+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0dc245-3643-6576-8001-0cd5e51f0f39'}}, tasks=(), interrupts=())

When interacting with the saved graph state, you must specify a thread identifier. You can view the latest state of the graph by calling graph.get_state(config). This will return a StateSnapshot object

StateSnapshot(
    values={'foo': 'b', 'bar': ['a', 'b']},
    next=(),
    config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28fe-6528-8002-5a559208592c'}},
    metadata={'source': 'loop', 'writes': {'node_b': {'foo': 'b', 'bar': ['b']}}, 'step': 2},
    created_at='2024-08-29T19:19:38.821749+00:00',
    parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1ef663ba-28f9-6ec4-8001-31981c2c39f8'}}, tasks=()
)

In [13]:
"""You can get the full history of the graph execution for a given thread by calling 
graph.get_state_history(config). """
config = {"configurable": {"thread_id": "1"}}
list(graph.get_state_history(config))

[StateSnapshot(values={'foo': 'b', 'bar': 'b'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0dc245-364d-6625-8002-5acb00f7c405'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-12-18T15:15:02.453610+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0dc245-3643-6576-8001-0cd5e51f0f39'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': 'a'}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0dc245-3643-6576-8001-0cd5e51f0f39'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-12-18T15:15:02.449496+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0dc245-3639-6c5c-8000-cd68ed475043'}}, tasks=(PregelTask(id='d2d3a34a-ed5e-989b-55ee-b89e08eb626a', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts=(), stat